# Bathymetry — shaded relief

Render a GEBCO 2020 subset of the Mid-Atlantic Ridge near the Azores as a
**hillshaded** relief map. Hillshading turns a flat depth grid into an
intuitive 3D-looking surface that makes the rift valley and transform faults
pop — exactly how a marine geologist would eyeball seafloor structure.

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from earthlens.core import EarthLens
from matplotlib.colors import LightSource
from pyramids.dataset import Dataset

In [ ]:
out = tempfile.mkdtemp()
paths = EarthLens(
    data_source='gebco',
    dataset='gebco_2020',
    aoi=[-29.5, 36.2, -27.7, 38.0],
    path=out,
).download(progress_bar=False)

arr = np.asarray(Dataset.read_file(str(paths[0])).read_array(), dtype='float32')
arr = arr[0] if arr.ndim == 3 else arr
arr = np.where(arr == 32767, np.nan, arr)
# LightSource needs finite values; fill the (rare) gaps with the deepest value.
filled = np.where(np.isnan(arr), np.nanmin(arr), arr)
print(
    'grid',
    arr.shape,
    '| depth range',
    round(np.nanmin(arr)),
    '..',
    round(np.nanmax(arr)),
    'm',
)

## Hillshade

Light from the NW (`azdeg=315`) at 45 degrees elevation; `dx`/`dy` are the
approximate metres-per-pixel of a 15-arcsec grid so the relief is physically
scaled rather than wildly exaggerated.

In [ ]:
ls = LightSource(azdeg=315, altdeg=45)
rgb = ls.shade(
    filled, cmap=plt.cm.gist_earth, vert_exag=1.0, dx=460, dy=460, blend_mode='soft'
)
plt.figure(figsize=(6, 6))
plt.imshow(rgb)
plt.title('GEBCO 2020 shaded relief — Mid-Atlantic Ridge (Azores)')
plt.axis('off')
plt.show()